# Chapter 3: Feature Engineering for NLP

Feature engineering is a critical step in any machine learning pipeline, and it is especially important in Natural Language Processing (NLP). In this chapter, we explore various techniques for transforming text data into numerical features that can be used by machine learning algorithms.

**Topics covered:**
1. Bag of Words (BoW)
2. TF-IDF (Term Frequency-Inverse Document Frequency)
3. Word Embeddings (Word2Vec and GloVe)
4. Introduction to BERT Embeddings

The goal is to convert text into numerical representations while preserving the underlying meaning and structure.

## Setup: Install Required Packages

Run the cell below to install all the libraries needed for this chapter.

In [ ]:
!pip install scikit-learn nltk gensim transformers torch --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

---
## 3.1 Bag of Words (BoW)

The **Bag of Words** model is a fundamental method for text representation in NLP. It converts text into numerical features by treating each document as an unordered collection of words, ignoring grammar, word order, and context, but retaining the frequency of each word.

### How it works (3 steps):
1. **Tokenizing the Text** -- Split text into individual words (tokens)
2. **Building a Vocabulary** -- Create a set of all unique words across the corpus
3. **Vectorizing the Text** -- Represent each document as a vector of word counts based on the vocabulary

### 3.1.1 Understanding the Bag of Words Model

Consider these two documents:
- Document 1: *"Natural language processing is fun"*
- Document 2: *"Language models are important in NLP"*

**Step 1 -- Tokenization:** Split each document into words.

**Step 2 -- Build Vocabulary:** Collect all unique words:
`["natural", "language", "processing", "is", "fun", "models", "are", "important", "in", "nlp"]`

**Step 3 -- Vectorize:** Represent each document as a count vector:
- Document 1: `[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]`
- Document 2: `[0, 1, 0, 0, 0, 1, 1, 1, 1, 1]`

Each position corresponds to a word in the vocabulary, and the value is the count of that word in the document.

### 3.1.2 Implementing Bag of Words in Python

We use `CountVectorizer` from scikit-learn to convert text into a BoW representation.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Sample text corpus
documents = [
    "Natural language processing is fun",
    "Language models are important in NLP"
]

# Initialize the CountVectorizer
vectorizer = CountVectorizer()

# Fit the vectorizer on the text data and transform to BoW matrix
X = vectorizer.fit_transform(documents)

# Convert the sparse matrix to a dense array
bow_array = X.toarray()

# Get the feature names (vocabulary)
vocab = vectorizer.get_feature_names_out()

print("Vocabulary:")
print(vocab)
print("\nBag of Words Array:")
print(bow_array)

**How it works:**
- `CountVectorizer()` learns the vocabulary from the text and transforms documents into a matrix of word counts.
- Each row in the output array corresponds to a document.
- Each column corresponds to a word in the vocabulary.
- The value at position `[i, j]` is the count of word `j` in document `i`.

### 3.1.3 Advantages and Limitations of BoW

**Advantages:**
- **Simplicity:** Easy to understand and implement
- **Efficiency:** Computationally efficient for small to medium corpora
- **Baseline:** Serves as a solid baseline for more complex models

**Limitations:**
- **Loss of Context:** Ignores word order and grammar
- **High Dimensionality:** Vocabulary size grows with the corpus, leading to large feature vectors
- **Sparsity:** Most elements in the vectors are zero

### 3.1.4 Practical Example: Text Classification with Bag of Words

Let's build a simple text classifier using BoW features and a Naive Bayes classifier.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Sample text corpus and labels
documents = [
    "Natural language processing is fun",
    "Language models are important in NLP",
    "I enjoy learning about artificial intelligence",
    "Machine learning and NLP are closely related",
    "Deep learning is a subset of machine learning"
]
labels = [1, 1, 0, 1, 0]  # 1 for NLP-related, 0 for AI-related

# Initialize the CountVectorizer
vectorizer = CountVectorizer()

# Transform the text data into BoW features
X = vectorizer.fit_transform(documents)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

# Initialize and train the Naive Bayes classifier
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

# Predict the labels for the test set
y_pred = classifier.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

**Workflow summary:**
1. Convert raw text into numerical features with `CountVectorizer`
2. Split the data into training and testing sets
3. Train a machine learning model (Naive Bayes) on the training data
4. Evaluate the model's performance on the test data

This same workflow applies to many text classification tasks: sentiment analysis, spam detection, document categorization, etc.

---
## 3.2 TF-IDF (Term Frequency-Inverse Document Frequency)

**TF-IDF** is a more sophisticated feature extraction technique than BoW. Unlike BoW, which simply counts word occurrences, TF-IDF takes into account the **importance** of each word relative to the entire corpus.

- Words that are **frequent in a document but rare across the corpus** get higher weights (they are more informative).
- Words that appear **frequently across many documents** (e.g., "the", "is", "and") get lower weights.

### 3.2.1 Understanding TF-IDF

TF-IDF is composed of two components:

**Term Frequency (TF):** Measures how frequently a term appears in a document.

$$TF(t, d) = \frac{\text{count of term } t \text{ in document } d}{\text{total terms in document } d}$$

**Inverse Document Frequency (IDF):** Measures how important a term is across the entire corpus.

$$IDF(t) = \log\frac{\text{total number of documents}}{\text{number of documents containing } t}$$

**TF-IDF score:**

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

Words that are unique to a document get high TF-IDF scores; common words across many documents get low scores.

### 3.2.2 Advantages of TF-IDF

- **Importance Weighting:** Highlights significant words while downplaying common ones
- **Improved Feature Representation:** More nuanced than raw word counts
- **Versatility:** Applicable to information retrieval, text classification, clustering, and more
- **Better Classification Performance:** Often outperforms plain BoW in ML tasks

### 3.2.3 Implementing TF-IDF in Python

We use `TfidfVectorizer` from scikit-learn, which works similarly to `CountVectorizer` but produces TF-IDF weighted values.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample text corpus
documents = [
    "Natural language processing is fun",
    "Language models are important in NLP",
    "I enjoy learning about artificial intelligence",
    "Machine learning and NLP are closely related",
    "Deep learning is a subset of machine learning"
]

# Initialize the TfidfVectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the text data
X = vectorizer.fit_transform(documents)

# Convert to dense array
tfidf_array = X.toarray()

# Get the feature names (vocabulary)
vocab = vectorizer.get_feature_names_out()

print("Vocabulary:")
print(vocab)
print("\nTF-IDF Array (first 2 documents):")
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print(tfidf_array[:2])

Notice how the TF-IDF values differ from simple word counts:
- Words that appear in only one document get higher values (they are more distinctive).
- Words that appear across multiple documents get lower values.

Compare the word "is" (appears in multiple documents) vs. "fun" (appears in only one) -- "fun" receives a higher TF-IDF weight.

### 3.2.4 Practical Example: Text Classification with TF-IDF

The same classification workflow as BoW, but using TF-IDF features instead.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Sample text corpus and labels
documents = [
    "Natural language processing is fun",
    "Language models are important in NLP",
    "I enjoy learning about artificial intelligence",
    "Machine learning and NLP are closely related",
    "Deep learning is a subset of machine learning"
]
labels = [1, 1, 0, 1, 0]  # 1 for NLP-related, 0 for AI-related

# Initialize the TfidfVectorizer
vectorizer = TfidfVectorizer()

# Transform the text data
X = vectorizer.fit_transform(documents)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

# Initialize and train the classifier
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

# Predict and evaluate
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("TF-IDF Classification Accuracy:", accuracy)

### 3.2.5 Comparing Bag of Words and TF-IDF

| Feature | Bag of Words | TF-IDF |
|---|---|---|
| **Method** | Raw word counts | Weighted word importance |
| **Common words** | Treated equally | Downweighted |
| **Rare/unique words** | Treated equally | Upweighted |
| **Simplicity** | Very simple | Slightly more complex |
| **Performance** | Good baseline | Often better for classification |
| **Context awareness** | None | None (still treats words independently) |

Both methods share the **sparsity** limitation -- for large vocabularies, the feature vectors contain mostly zeros. Neither captures word order or semantic meaning.

---
## 3.3 Word Embeddings (Word2Vec, GloVe)

**Word embeddings** represent words as dense vectors in a continuous vector space. Unlike BoW and TF-IDF which produce sparse, high-dimensional vectors, word embeddings:

- Capture **semantic relationships** between words (similar words have similar vectors)
- Are **low-dimensional** and **dense** (typically 50-300 dimensions)
- Support **mathematical operations** on meaning (e.g., king - man + woman = queen)
- Enable **transfer learning** -- pre-trained embeddings can be reused across tasks

### 3.3.1 Understanding Word Embeddings

Word embeddings map words to vectors of real numbers in a low-dimensional space. The key idea is that words used in similar contexts end up with similar vector representations.

**Key concepts:**
- **Semantic Similarity:** "king" and "queen" have similar vectors because they appear in similar contexts.
- **Continuous Vector Space:** Words can be compared using vector arithmetic. The classic example: `vector("king") - vector("man") + vector("woman") ~ vector("queen")`
- **Dimensionality Reduction:** Word embeddings compress meaning into far fewer dimensions than BoW/TF-IDF.

**Two popular methods:**
1. **Word2Vec** (Google) -- learns embeddings by predicting words from their local context
2. **GloVe** (Stanford) -- learns embeddings from global word co-occurrence statistics

### 3.3.2 Word2Vec

Word2Vec comes in two variants:

1. **Continuous Bag of Words (CBOW):** Predicts the target word given the surrounding context words.
2. **Skip-Gram:** Predicts the surrounding context words given a target word.

Both approaches learn vector representations that capture word relationships based on local context windows.

Let's train a Word2Vec model using the Gensim library.

In [ ]:
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize, word_tokenize

# Sample text corpus
text = ("Natural language processing is fun and exciting. "
        "Language models are important in NLP. "
        "I enjoy learning about artificial intelligence. "
        "Machine learning and NLP are closely related. "
        "Deep learning is a subset of machine learning.")

# Tokenize the text into sentences, then each sentence into words
sentences = sent_tokenize(text)
tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]

print("Tokenized sentences:")
for i, sent in enumerate(tokenized_sentences):
    print(f"  Sentence {i+1}: {sent}")

In [ ]:
# Train a Word2Vec model using the Skip-Gram method
model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,  # Dimensionality of the word vectors
    window=5,         # Max distance between current and predicted word
    sg=1,             # 1 = Skip-Gram, 0 = CBOW
    min_count=1       # Include all words (even those appearing once)
)

# Get the vector representation of the word "language"
vector = model.wv['language']
print("Vector representation of 'language' (first 10 dimensions):")
print(vector[:10])
print(f"\nFull vector shape: {vector.shape}")

In [ ]:
# Find the most similar words to "language"
similar_words = model.wv.most_similar('language')
print("Most similar words to 'language':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

**Key parameters explained:**
- `vector_size=100`: Each word is represented by a 100-dimensional vector
- `window=5`: Context window of 5 words to the left and right
- `sg=1`: Use Skip-Gram (set to `0` for CBOW)
- `min_count=1`: Include words that appear at least once

> **Note:** With a small corpus like this, the similarity scores may not be very meaningful. Word2Vec works best with large amounts of text data.

### 3.3.3 GloVe (Global Vectors for Word Representation)

**GloVe** was developed at Stanford and takes a different approach from Word2Vec:

- It constructs a large **word co-occurrence matrix** capturing how often word pairs appear together across the entire corpus.
- It then applies **matrix factorization** to learn dense word vectors.
- This captures both **local** (within-window) and **global** (corpus-wide) statistical information.

We can load pre-trained GloVe embeddings (trained on Wikipedia + Gigaword) via Gensim's downloader API.

In [ ]:
import gensim.downloader as api

# Load pre-trained GloVe embeddings (100-dimensional vectors)
# This download may take a few minutes the first time
print("Loading GloVe embeddings (this may take a minute)...")
glove_model = api.load("glove-wiki-gigaword-100")
print("Done! Vocabulary size:", len(glove_model))

In [ ]:
# Get the vector representation of the word "language"
vector = glove_model['language']
print("Vector representation of 'language' (first 10 dimensions):")
print(vector[:10])
print(f"Full vector shape: {vector.shape}")

In [ ]:
# Find the most similar words to "language"
similar_words = glove_model.most_similar('language')
print("Most similar words to 'language':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

In [ ]:
# Demonstrate word analogy: king - man + woman = ?
result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print("king - man + woman = ?")
for word, score in result:
    print(f"  {word}: {score:.4f}")

### 3.3.4 Comparing Word2Vec and GloVe

| Feature | Word2Vec | GloVe |
|---|---|---|
| **Developer** | Google | Stanford |
| **Approach** | Predict context words (local) | Matrix factorization of co-occurrence (global) |
| **Context** | Local context windows | Both local and global statistics |
| **Training speed** | Fast on large datasets | Efficient, parallelizable |
| **Best for** | Large-scale, domain-specific training | Leveraging pre-trained embeddings |

Both produce high-quality word embeddings. The choice often depends on your specific use case and available data.

### 3.3.5 Advantages and Limitations of Word Embeddings

**Advantages:**
- **Semantic Representation:** Capture meaningful relationships between words
- **Compact Representation:** Low-dimensional, dense vectors (vs. sparse BoW/TF-IDF)
- **Transfer Learning:** Pre-trained embeddings can be reused across tasks

**Limitations:**
- **Out-of-Vocabulary (OOV) Words:** Cannot represent words not seen during training
- **Context Ignorance:** A word always has the same embedding regardless of context (e.g., "bank" in "river bank" vs. "bank account" gets the same vector)

---
## 3.4 Introduction to BERT Embeddings

**BERT** (Bidirectional Encoder Representations from Transformers) is a state-of-the-art model developed by Google that generates **context-aware embeddings**.

Unlike Word2Vec and GloVe, which assign a single fixed vector to each word, BERT produces different embeddings for the same word depending on its surrounding context. For example, "bank" will have different embeddings in:
- *"I went to the river bank"*
- *"I deposited money at the bank"*

### 3.4.1 Understanding BERT

**Key features:**

1. **Bidirectional Context:** BERT considers both the left and right context of a word simultaneously, unlike traditional left-to-right or right-to-left models.

2. **Pre-trained Models:** BERT is pre-trained on massive datasets (Wikipedia + BooksCorpus) using two tasks:
   - **Masked Language Modeling (MLM):** Randomly masks words and predicts them from context
   - **Next Sentence Prediction (NSP):** Predicts whether two sentences are consecutive

3. **Transformer Architecture:** Uses multi-layer self-attention to capture complex word relationships.

4. **Fine-tuning:** The pre-trained model can be adapted to specific tasks (classification, NER, QA, etc.) with relatively small labeled datasets.

### 3.4.2 How BERT Works

BERT uses a two-step approach:

**Step 1 -- Pre-training:**
- **Masked Language Modeling (MLM):** Randomly masks 15% of tokens and trains the model to predict them. E.g., *"The quick brown [MASK] jumps over the lazy dog"* -> predicts *"fox"*.
- **Next Sentence Prediction (NSP):** Given two sentences, predicts if the second follows the first.

**Step 2 -- Fine-tuning:**
- Take the pre-trained BERT model and add a task-specific output layer.
- Train on task-specific labeled data (e.g., sentiment labels for sentiment analysis).
- The pre-trained weights provide a strong starting point, requiring less task-specific data.

### 3.4.3 Implementing BERT Embeddings in Python

We use the `transformers` library from Hugging Face to generate BERT embeddings.

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
print("Loading BERT model (this may take a moment)...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
print("Done!")

# Sample text
text = "Natural Language Processing is fascinating."

# Tokenize the text (returns PyTorch tensors)
inputs = tokenizer(text, return_tensors='pt')
print("\nTokenized input IDs:", inputs['input_ids'])
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

In [ ]:
# Generate BERT embeddings (no gradient computation needed for inference)
with torch.no_grad():
    outputs = model(**inputs)

# Get the embeddings for the [CLS] token
# The [CLS] token represents the aggregate meaning of the entire input
cls_embeddings = outputs.last_hidden_state[:, 0, :]

print("BERT [CLS] embedding shape:", cls_embeddings.shape)
print("\nFirst 20 values:")
print(cls_embeddings[0, :20])

**How it works:**
1. The tokenizer converts text into token IDs that BERT understands (including special tokens `[CLS]` and `[SEP]`).
2. The model processes the tokens through multiple Transformer layers.
3. `outputs.last_hidden_state` contains embeddings for every token.
4. The `[CLS]` token (index 0) is commonly used as a sentence-level representation.

Each embedding is a 768-dimensional vector (for `bert-base-uncased`).

### 3.4.4 Fine-tuning BERT for Text Classification

BERT can be fine-tuned for specific tasks by adding a classification layer on top. The example below shows the full workflow using Hugging Face's `Trainer` API.

> **Note:** Fine-tuning BERT is computationally intensive. A GPU is recommended for larger datasets. This example uses a tiny dataset for demonstration purposes.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch

# Sample text corpus and labels
documents = [
    "Natural Language Processing is fascinating.",
    "Machine learning models are essential for AI.",
    "I love learning about deep learning.",
    "NLP and AI are closely related fields.",
    "Artificial Intelligence is transforming industries."
]
labels = [1, 0, 1, 1, 0]  # 1 for NLP-related, 0 for AI-related

# Load pre-trained BERT tokenizer and classification model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Tokenize all documents
inputs = tokenizer(documents, padding=True, truncation=True, return_tensors='pt')

print("Tokenized shapes:")
for key, val in inputs.items():
    print(f"  {key}: {val.shape}")

In [ ]:
# Create a custom PyTorch Dataset
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# For this tiny dataset, we use all data for training and a subset for eval
train_dataset = TextDataset(inputs, labels)
eval_dataset = TextDataset(inputs, labels)  # Same data for demo purposes

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    report_to='none',  # Disable wandb/other logging
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Train the model
print("Training BERT for text classification...")
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print("\nEvaluation results:")
print(results)

### 3.4.5 Advantages and Limitations of BERT

**Advantages:**
- **Context-Aware Embeddings:** Different representations for the same word in different contexts
- **State-of-the-Art Performance:** Achieves top results on many NLP benchmarks
- **Transfer Learning:** Pre-trained models can be fine-tuned on small task-specific datasets

**Limitations:**
- **Computationally Intensive:** Requires significant resources (GPU/TPU recommended)
- **Complexity:** More complex architecture compared to traditional embeddings
- **Model Size:** BERT-base has 110M parameters; larger variants have even more

---
## Summary: Feature Engineering Techniques Compared

| Technique | Type | Dimensions | Context-Aware | Semantic Similarity | Complexity |
|---|---|---|---|---|---|
| **Bag of Words** | Sparse | Vocabulary size | No | No | Low |
| **TF-IDF** | Sparse | Vocabulary size | No | No | Low |
| **Word2Vec** | Dense | 50-300 | No | Yes | Medium |
| **GloVe** | Dense | 50-300 | No | Yes | Medium |
| **BERT** | Dense | 768+ | Yes | Yes | High |

---
## Practical Exercises

### Exercise 1: Bag of Words

**Task:** Use `CountVectorizer` to transform the following corpus into a Bag of Words representation. Print the vocabulary and the BoW array.

```python
documents = [
    "Text processing is important for NLP.",
    "Bag of Words is a simple text representation method.",
    "Feature engineering is essential in machine learning."
]
```

In [ ]:
# Exercise 1: Your solution here
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "Text processing is important for NLP.",
    "Bag of Words is a simple text representation method.",
    "Feature engineering is essential in machine learning."
]

# TODO: Initialize CountVectorizer, fit_transform, and print results


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "Text processing is important for NLP.",
    "Bag of Words is a simple text representation method.",
    "Feature engineering is essential in machine learning."
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
bow_array = X.toarray()
vocab = vectorizer.get_feature_names_out()

print("Vocabulary:")
print(vocab)
print("\nBag of Words Array:")
print(bow_array)
```
</details>

### Exercise 2: TF-IDF

**Task:** Use `TfidfVectorizer` to transform the following corpus into a TF-IDF representation. Print the vocabulary and the TF-IDF array.

```python
documents = [
    "Natural language processing is fun.",
    "Language models are important in NLP.",
    "Machine learning and NLP are closely related."
]
```

In [ ]:
# Exercise 2: Your solution here
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "Natural language processing is fun.",
    "Language models are important in NLP.",
    "Machine learning and NLP are closely related."
]

# TODO: Initialize TfidfVectorizer, fit_transform, and print results


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "Natural language processing is fun.",
    "Language models are important in NLP.",
    "Machine learning and NLP are closely related."
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)
tfidf_array = X.toarray()
vocab = vectorizer.get_feature_names_out()

print("Vocabulary:")
print(vocab)
print("\nTF-IDF Array:")
print(tfidf_array)
```
</details>

### Exercise 3: Word2Vec

**Task:** Train a Word2Vec model on the following text and obtain the vector representation of the word "NLP". Also find the most similar words.

```python
text = ("Natural language processing is fun and exciting. "
        "Language models are important in NLP. "
        "Machine learning and NLP are closely related.")
```

In [ ]:
# Exercise 3: Your solution here
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize, word_tokenize

text = ("Natural language processing is fun and exciting. "
        "Language models are important in NLP. "
        "Machine learning and NLP are closely related.")

# TODO: Tokenize, train Word2Vec, get vector for 'NLP', find similar words


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize, word_tokenize

text = ("Natural language processing is fun and exciting. "
        "Language models are important in NLP. "
        "Machine learning and NLP are closely related.")

sentences = sent_tokenize(text)
tokenized_sentences = [word_tokenize(s) for s in sentences]

model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, sg=1, min_count=1)

vector = model.wv['NLP']
print("Vector representation of 'NLP' (first 10 dims):")
print(vector[:10])

similar = model.wv.most_similar('NLP')
print("\nMost similar words to 'NLP':")
for word, score in similar:
    print(f"  {word}: {score:.4f}")
```
</details>

### Exercise 4: GloVe

**Task:** Load the pre-trained GloVe embeddings (`glove-wiki-gigaword-100`) and find the most similar words to "machine". Also try a word analogy of your choice.

In [ ]:
# Exercise 4: Your solution here
import gensim.downloader as api

# TODO: Load GloVe, find similar words to 'machine', try a word analogy


<details>
<summary><b>Click to reveal solution</b></summary>

```python
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-100")

vector = glove_model['machine']
print("Vector representation of 'machine' (first 10 dims):")
print(vector[:10])

similar = glove_model.most_similar('machine')
print("\nMost similar words to 'machine':")
for word, score in similar:
    print(f"  {word}: {score:.4f}")

# Word analogy: king - man + woman = queen
result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print("\nking - man + woman = ?")
for word, score in result:
    print(f"  {word}: {score:.4f}")
```
</details>

### Exercise 5: BERT Embeddings

**Task:** Use the `transformers` library to generate BERT embeddings for the following text. Print the shape and first 20 values of the `[CLS]` token embedding.

```python
text = "Transformers are powerful models for NLP tasks."
```

In [ ]:
# Exercise 5: Your solution here
from transformers import BertTokenizer, BertModel
import torch

text = "Transformers are powerful models for NLP tasks."

# TODO: Load BERT, tokenize, generate embeddings, print [CLS] embedding


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from transformers import BertTokenizer, BertModel
import torch

text = "Transformers are powerful models for NLP tasks."

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

cls_embeddings = outputs.last_hidden_state[:, 0, :]
print("BERT [CLS] embedding shape:", cls_embeddings.shape)
print("\nFirst 20 values:")
print(cls_embeddings[0, :20])
```
</details>